# 🔧 Fine-tune viXTTS trên giọng tiếng Việt riêng

Notebook chạy trên **Google Colab Pro** (GPU T4 / L4 / A100).

Fine-tune **tiếp** từ checkpoint [`capleaf/viXTTS`](https://huggingface.co/capleaf/viXTTS) (đã có tokenizer tiếng Việt) trên giọng của bạn.

> ⚠️ **Chỉ dùng với giọng của chính bạn hoặc giọng có đồng ý hợp pháp.** Mục đích: học tập / nghiên cứu.

**Các bước:** kiểm tra GPU → cài đặt → mount Drive → chuẩn bị dữ liệu → (tùy chọn) tạo transcript bằng Whisper → cấu hình → train → kiểm thử → xuất model.

## 1. Kiểm tra GPU
Vào **Runtime → Change runtime type → GPU** trước khi chạy.

In [ ]:
!nvidia-smi

## 2. Cài đặt
Cài Coqui TTS (fork idiap được maintain) + công cụ phụ trợ. Toàn bộ nằm trong runtime Colab, không ảnh hưởng máy bạn.

In [ ]:
# Coqui TTS + tiện ích. Colab đã có sẵn torch hợp với CUDA của runtime.
!pip install -q coqui-tts==0.27.2 faster-whisper==1.0.3 huggingface_hub
# Chuẩn hóa text tiếng Việt (tùy chọn, giúp đọc số/ký hiệu tự nhiên hơn)
!pip install -q vinorm deepfilternet || echo 'Bo qua goi tuy chon neu loi'
print('Cai dat xong.')

## 3. Mount Google Drive
Lưu dataset + checkpoint vào Drive để không mất khi Colab ngắt phiên.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/vixtts_finetune'
os.makedirs(WORK_DIR, exist_ok=True)
print('Thu muc lam viec:', WORK_DIR)

## 4. Tải checkpoint viXTTS gốc
Đây là điểm khởi đầu để fine-tune tiếp.

In [ ]:
from huggingface_hub import snapshot_download

BASE_MODEL_DIR = '/content/vixtts_base'
snapshot_download(repo_id='capleaf/viXTTS', local_dir=BASE_MODEL_DIR,
                  local_dir_use_symlinks=False)
print('Da tai viXTTS goc ve:', BASE_MODEL_DIR)
!ls -lh {BASE_MODEL_DIR}

## 5. Chuẩn bị dữ liệu

**Cách A — đã có dataset chuẩn:** nén thành `dataset.zip` theo cấu trúc bên dưới rồi upload vào `WORK_DIR` trên Drive.

```
dataset/
├── metadata.csv      # dòng: wavs/0001.wav|noi dung loi|speaker1
└── wavs/*.wav
```

**Cách B — chỉ có vài file audio dài, chưa có transcript:** dùng cell Whisper ở mục 6 để tự cắt đoạn + sinh transcript.

In [ ]:
# Cách A: giải nén dataset.zip da upload san vao WORK_DIR
import os, zipfile
DATASET_DIR = '/content/dataset'
zip_path = os.path.join(WORK_DIR, 'dataset.zip')
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path) as z:
        z.extractall('/content')
    print('Da giai nen dataset vao', DATASET_DIR)
    !head -n 5 {DATASET_DIR}/metadata.csv
else:
    print('Chua thay dataset.zip. Dung Cach B (Whisper) o muc 6.')

## 6. (Tùy chọn) Tự tạo dataset từ audio dài bằng Whisper
Upload các file audio dài (giọng bạn) vào `WORK_DIR/raw_audio/` rồi chạy cell này. Nó sẽ phiên âm, cắt theo câu, và tạo `metadata.csv`.

In [ ]:
import os, glob, csv
import soundfile as sf
from faster_whisper import WhisperModel

RAW_DIR = os.path.join(WORK_DIR, 'raw_audio')
DATASET_DIR = '/content/dataset'
WAVS_DIR = os.path.join(DATASET_DIR, 'wavs')
os.makedirs(WAVS_DIR, exist_ok=True)

raw_files = sorted(glob.glob(os.path.join(RAW_DIR, '*')))
assert raw_files, f'Khong tim thay file nao trong {RAW_DIR}. Hay upload audio truoc.'

# 'large-v3' chinh xac nhat; doi 'medium' neu muon nhanh hon
model = WhisperModel('large-v3', device='cuda', compute_type='float16')

rows = []
idx = 0
for f in raw_files:
    segments, _ = model.transcribe(f, language='vi', vad_filter=True)
    audio, sr = sf.read(f)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)  # ve mono
    for seg in segments:
        text = seg.text.strip()
        if len(text) < 3:
            continue
        s, e = int(seg.start * sr), int(seg.end * sr)
        clip = audio[s:e]
        if len(clip) < sr * 0.5:   # bo doan qua ngan (<0.5s)
            continue
        idx += 1
        name = f'wavs/{idx:04d}.wav'
        sf.write(os.path.join(DATASET_DIR, name), clip, sr)
        rows.append([name, text, 'speaker1'])

with open(os.path.join(DATASET_DIR, 'metadata.csv'), 'w', newline='', encoding='utf-8') as fp:
    csv.writer(fp, delimiter='|').writerows(rows)

print(f'Tao xong {len(rows)} doan tu {len(raw_files)} file audio.')
!head -n 5 {DATASET_DIR}/metadata.csv

## 7. Cấu hình & nạp dữ liệu cho training
Tách train/eval và nạp config XTTS từ checkpoint viXTTS gốc.

In [ ]:
import os, random
from TTS.tts.datasets import load_tts_samples
from TTS.tts.configs.shared_configs import BaseDatasetConfig

DATASET_DIR = '/content/dataset'
OUT_DIR = os.path.join(WORK_DIR, 'run')
os.makedirs(OUT_DIR, exist_ok=True)

dataset_config = BaseDatasetConfig(
    formatter='ljspeech',
    dataset_name='vi_custom',
    path=DATASET_DIR,
    meta_file_train='metadata.csv',
    language='vi',
)

train_samples, eval_samples = load_tts_samples(
    dataset_config, eval_split=True, eval_split_size=0.1,
)
print(f'Train: {len(train_samples)} | Eval: {len(eval_samples)}')

In [ ]:
import os
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts

# Nap config tu checkpoint viXTTS goc
config = XttsConfig()
config.load_json(os.path.join(BASE_MODEL_DIR, 'config.json'))

# Tham so train cho dataset nho
config.batch_size = 3
config.eval_batch_size = 3
config.num_loader_workers = 2
config.print_step = 25
config.save_step = 500
config.epochs = 10              # tang/giam tuy luong du lieu
config.output_path = OUT_DIR
config.run_eval = True
config.optimizer_wd_only_on_weights = True
config.datasets = [dataset_config]

print('Config san sang. epochs =', config.epochs, '| batch =', config.batch_size)

## 8. Train
Khởi tạo model từ checkpoint viXTTS rồi fine-tune. Checkpoint được lưu vào `WORK_DIR/run` trên Drive.

In [ ]:
import os
from trainer import Trainer, TrainerArgs

model = Xtts.init_from_config(config)
model.load_checkpoint(
    config,
    checkpoint_dir=BASE_MODEL_DIR,   # bat dau tu viXTTS
    use_deepspeed=False,
)

trainer = Trainer(
    TrainerArgs(restore_path=None, skip_train_epoch=False),
    config,
    output_path=OUT_DIR,
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples,
)
trainer.fit()

## 9. Kiểm thử nhanh
Sinh thử một câu bằng checkpoint vừa train. Sửa `CKPT_DIR` trỏ tới thư mục run mới nhất nếu cần.

In [ ]:
import glob, os, torch, torchaudio
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.models.xtts import Xtts
from IPython.display import Audio, display

# Tim thu muc checkpoint moi nhat
runs = sorted(glob.glob(os.path.join(OUT_DIR, '*')), key=os.path.getmtime)
CKPT_DIR = runs[-1]
print('Dung checkpoint:', CKPT_DIR)

cfg = XttsConfig(); cfg.load_json(os.path.join(BASE_MODEL_DIR, 'config.json'))
m = Xtts.init_from_config(cfg)
m.load_checkpoint(cfg, checkpoint_dir=CKPT_DIR, use_deepspeed=False)
m.cuda().eval()

# File giong mau de lay dac trung giong (lay 1 file trong dataset)
ref = sorted(glob.glob('/content/dataset/wavs/*.wav'))[0]
gpt_lat, spk_emb = m.get_conditioning_latents(audio_path=ref)
out = m.inference(
    text='Xin chào, đây là giọng nói đã được tinh chỉnh.',
    language='vi', gpt_cond_latent=gpt_lat, speaker_embedding=spk_emb,
    temperature=0.3, enable_text_splitting=True,
)
torchaudio.save('/content/test.wav', torch.tensor(out['wav']).unsqueeze(0), 24000)
display(Audio('/content/test.wav'))

## 10. Xuất model để dùng cho web app
Gom `model.pth`, `config.json`, `vocab.json` vào một thư mục rồi tải về. Copy vào `voice-clone-web/backend/model/`.

In [ ]:
import os, glob, shutil
EXPORT = os.path.join(WORK_DIR, 'export_model')
os.makedirs(EXPORT, exist_ok=True)

# model.pth tu checkpoint moi nhat
pths = sorted(glob.glob(os.path.join(CKPT_DIR, '*.pth')), key=os.path.getmtime)
shutil.copy(pths[-1], os.path.join(EXPORT, 'model.pth'))
# config + vocab lay tu base (tuong thich tokenizer tieng Viet)
shutil.copy(os.path.join(BASE_MODEL_DIR, 'config.json'), EXPORT)
shutil.copy(os.path.join(BASE_MODEL_DIR, 'vocab.json'), EXPORT)

print('Da xuat model vao Drive:', EXPORT)
!ls -lh {EXPORT}
print('\nTai thu muc nay ve va copy vao voice-clone-web/backend/model/')